In [0]:

from pyspark.sql import functions as F

base = "/Volumes/workspace/default/freddie_raw"

# Check uploaded files
display(dbutils.fs.ls(base))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/freddie_raw/sample_orig_2017.txt,sample_orig_2017.txt,6289093,1793476744000
dbfs:/Volumes/workspace/default/freddie_raw/sample_perf_2017.txt,sample_perf_2017.txt,322479831,1793476818000


In [0]:

# Read origination data
orig = (
    spark.read
    .option("sep", "|")
    .option("header", "false")
    .csv(f"{base}/sample_orig_2017.txt")
)

# Read monthly performance data
perf = (
    spark.read
    .option("sep", "|")
    .option("header", "false")
    .csv(f"{base}/sample_perf_2017.txt")
)

# Check dimensions
print("=== ORIGINATION DATA ===")
print(f"Rows: {orig.count():,}")
print(f"Columns: {len(orig.columns)}")

print("\n=== PERFORMANCE DATA ===")
print(f"Rows: {perf.count():,}")
print(f"Columns: {len(perf.columns)}")

# Preview records
display(orig.limit(5))
display(perf.limit(5))


=== ORIGINATION DATA ===
Rows: 50,000
Columns: 31

=== PERFORMANCE DATA ===
Rows: 2,837,053
Columns: 35


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29,_c30
809,201705,N,204704,null,0,1,P,75,38,195000,75,4.25,R,N,FRM,PA,SF,179,F17Q10000002,N,360,1,OTHER,N,null,null,N,7,N,9999
702,201703,N,203202,null,0,1,P,80,36,187000,76,3,R,N,FRM,KY,SF,420,F17Q10000017,N,180,2,OTHER,N,null,null,N,7,N,9999
792,201703,N,204702,null,0,1,S,60,36,87000,60,3.375,R,N,FRM,MI,SF,487,F17Q10000064,N,360,2,OTHER,N,null,null,N,7,N,9999
776,201703,N,204702,null,0,1,S,80,18,106000,80,4.25,R,N,FRM,NY,SF,127,F17Q10000065,P,360,2,OTHER,N,null,null,N,7,N,9999
790,201703,N,204702,41620,0,1,I,75,42,75000,75,4.25,C,N,FRM,UT,CO,841,F17Q10000073,C,360,1,OTHER,N,null,null,N,7,N,9999


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29,_c30,_c31,_c32,_c33,_c34
F17Q10000002,201704,195000.00,00,0,360,null,null,null,null,4.250,0.00,null,null,null,null,null,null,null,null,null,null,0.00,null,null,81,null,null,null,null,null,195000.00,7,OTHER,null
F17Q10000002,201705,194000.00,00,1,359,null,null,null,null,4.250,0.00,null,null,null,null,null,null,null,null,null,null,0.00,null,null,77,null,null,null,null,null,194000.00,7,OTHER,null
F17Q10000002,201706,194000.00,00,2,358,null,null,null,null,4.250,0.00,null,null,null,null,null,null,null,null,null,null,0.00,null,null,75,null,null,null,null,null,194000.00,7,OTHER,null
F17Q10000002,201707,194000.00,00,3,357,null,null,null,null,4.250,0.00,null,null,null,null,null,null,null,null,null,null,0.00,null,null,74,null,null,null,null,null,194000.00,7,OTHER,null
F17Q10000002,201708,193000.00,00,4,356,null,null,null,null,4.250,0.00,null,null,null,null,null,null,null,null,null,null,0.00,null,null,73,null,null,null,null,null,193000.00,7,OTHER,null


In [0]:
# Check loan IDs
orig_ids = orig.select(F.col("_c19").alias("loan_id")).distinct()
perf_ids = perf.select(F.col("_c0").alias("loan_id")).distinct()

print("Unique origination loans:", orig_ids.count())
print("Unique performance loans:", perf_ids.count())

# Performance IDs not found in origination
unmatched = perf_ids.join(
    orig_ids, "loan_id", "left_anti"
).count()

print("Unmatched loans:", unmatched)

# Delinquency status distribution
display(
    perf.groupBy("_c3")
        .count()
        .orderBy(F.desc("count"))
)

# Reporting period coverage
display(
    perf.agg(
        F.min("_c1").alias("first_period"),
        F.max("_c1").alias("last_period")
    )
)


Unique origination loans: 50000
Unique performance loans: 50000
Unmatched loans: 0


_c3,count
00,2771604
01,26350
02,8372
03,5067
04,3685
05,3197
06,2455
07,2075
08,1759
09,1601


first_period,last_period
201701,202606


In [0]:

orig.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.freddie_orig_raw")

perf.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.freddie_perf_raw")

print("Raw Delta tables created!")


Raw Delta tables created!
